# CombBandits — exp8_scaling_d (GPU)

Runs the large-d dimension scaling experiment on Colab GPU.
This is the paper's strongest figure: CUCB regret grows as √d, LLM-CUCB-AT stays flat.

**Before running:**
1. Runtime → Change runtime type → **A100 GPU** (Colab Premium)
2. Run all cells top-to-bottom
3. Results upload automatically to S3 when done

**Estimated time:** ~15 minutes on A100

In [ ]:
# ── Verify GPU ────────────────────────────────────────────────────────────────
import subprocess
result = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
print(result.stdout if result.returncode == 0 else 'No GPU detected — switch runtime to GPU!')

import torch
print(f'\nCUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
else:
    raise RuntimeError('Switch runtime type to GPU (Runtime > Change runtime type > A100)')

In [ ]:
# ── Clone repo ────────────────────────────────────────────────────────────────
!git clone https://github.com/vkmk1/CombBandits /content/CombBandits
%cd /content/CombBandits

In [ ]:
# ── Install dependencies ──────────────────────────────────────────────────────
!pip install -q uv
!uv pip install --system -q -e '.[dev]'
print('Install complete.')

In [ ]:
# ── Run exp8_scaling_d on GPU ─────────────────────────────────────────────────
import time
start = time.time()

!python -m combbandits.cli run-gpu \
    configs/experiments/exp8_scaling_d.yaml \
    --output-dir results/exp8_scaling_d \
    --device cuda

elapsed = time.time() - start
print(f'\nexp8 done in {elapsed/60:.1f} min')

In [ ]:
# ── Generate metrics + figures ────────────────────────────────────────────────
!python -m combbandits.cli metrics results/exp8_scaling_d/exp8_scaling_d_results.json
!python -m combbandits.cli plot results/exp8_scaling_d/exp8_scaling_d_results.json \
    --output-dir figures/exp8_scaling_d
print('Figures saved to figures/exp8_scaling_d/')

In [ ]:
# ── Upload results to S3 ──────────────────────────────────────────────────────
# Install boto3 and configure AWS credentials to push results to S3.
# Paste your AWS credentials below (or use Colab Secrets).
import os

# ── Fill these in ──
AWS_ACCESS_KEY_ID     = ''   # your_access_key_id
AWS_SECRET_ACCESS_KEY = ''   # your_secret_access_key
AWS_REGION            = 'us-east-1'
S3_BUCKET             = 'combbandits-results-099841456154'
# ──────────────────

if not AWS_ACCESS_KEY_ID:
    print('⚠  No credentials set — skipping S3 upload.')
    print('   Download manually: Files panel → results/exp8_scaling_d/')
else:
    os.environ['AWS_ACCESS_KEY_ID']     = AWS_ACCESS_KEY_ID
    os.environ['AWS_SECRET_ACCESS_KEY'] = AWS_SECRET_ACCESS_KEY
    os.environ['AWS_DEFAULT_REGION']    = AWS_REGION

    import boto3
    s3 = boto3.client('s3')

    for local_path, s3_prefix in [
        ('results/exp8_scaling_d', 'results/exp8_scaling_d'),
        ('figures/exp8_scaling_d', 'figures/exp8_scaling_d'),
    ]:
        for root, dirs, files in os.walk(local_path):
            for fname in files:
                full = os.path.join(root, fname)
                key  = full.replace(local_path, s3_prefix, 1).lstrip('/')
                s3.upload_file(full, S3_BUCKET, key)
                print(f'  uploaded s3://{S3_BUCKET}/{key}')

    print('\nS3 upload complete.')

In [ ]:
# ── Preview: regret vs dimension plot ─────────────────────────────────────────
import json, numpy as np, matplotlib.pyplot as plt, os

with open('results/exp8_scaling_d/exp8_scaling_d_results.json') as f:
    results = json.load(f)

from collections import defaultdict
data = defaultdict(list)  # agent -> list of (d, final_regret)
for r in results:
    data[r['agent']].append((r['d'], r['final_regret']))

fig, ax = plt.subplots(figsize=(8, 5))
colors = {'cucb': 'steelblue', 'cts': 'green', 'llm_cucb_at': 'crimson', 'llm_greedy': 'orange'}

for agent, points in sorted(data.items()):
    by_d = defaultdict(list)
    for d, r in points:
        by_d[d].append(r)
    ds = sorted(by_d.keys())
    means = [np.mean(by_d[d]) for d in ds]
    ax.plot(ds, means, 'o-', label=agent, color=colors.get(agent))

ax.set_xlabel('Ground set size d', fontsize=12)
ax.set_ylabel('Mean cumulative regret (T=100K)', fontsize=12)
ax.set_title('Regret vs Dimension — exp8_scaling_d', fontsize=13)
ax.legend()
ax.set_xscale('log')
ax.set_yscale('log')
plt.tight_layout()
plt.savefig('figures/exp8_scaling_d/regret_vs_d_preview.png', dpi=150)
plt.show()
print('Key result: LLM-CUCB-AT regret should be flat; CUCB should grow with sqrt(d)')